In [ ]:
:set -XRankNTypes
:set -XTypeOperators
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XInstanceSigs
:set -XScopedTypeVariables
:set -XMultiParamTypeClasses
:set -XFunctionalDependencies
import Data.List (intercalate, sort, nub)
import Data.Maybe (mapMaybe, catMaybes)
putStrLn "Setup done." 

# 🔺 Иерархия функторов в Haskell

Иерархия функторов Haskell простирается далеко за пределы базового класса `Functor`.
Этот ноутбук исследует полную картину: от Functor до Selective.

> **📦 Зависимости**
> **Пакеты:** `base`
> **Расширения:** `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `FunctionalDependencies` — зависимости параметров класса: | m -> s ([→](Extensions.ipynb#multiparamtypeclasses)) · `InstanceSigs` — сигнатуры методов прямо в instance ([→](Extensions.ipynb#instancesigs)) · `MultiParamTypeClasses` — классы с несколькими параметрами ([→](Extensions.ipynb#multiparamtypeclasses)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables)) · `TypeOperators` — операторы на уровне типов: f :+: g ([→](Extensions.ipynb#typeoperators))


❖Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | 1️⃣ `Functor` — эндофунктор в Hask |  |
| 2 | 2️⃣ `Contravariant` — функторы, обращающие стрелки |  |
| 3 | 3️⃣ `Bifunctor` и `Bitraversable` |  |
| 4 | 4️⃣ `Invariant` — функторы в обоих направлениях |  |
| 5 | 5️⃣ `Applicative` как моноидальный функтор |  |
| 6 | 6️⃣ `Alternative` — моноид над аппликативными |  |
| 7 | 7️⃣ `Selective` — между Applicative и Monad |  |
| 8 | 8️⃣ Полная иерархия |  |

## 1️⃣ `Functor` — эндофунктор в Hask

**Функтор** `f :: * -> *` — это эндофунктор категории `Hask`:

```
class Functor f where
  fmap :: (a -> b) -> f a -> f b
```

Законы:
- `fmap id = id`
- `fmap (f . g) = fmap f . fmap g`

Категориальная интерпретация: `fmap` отображает морфизмы Hask в морфизмы Hask.

In [ ]:
-- Functor laws: inline verification (no polymorphic IO helper)
let f = (+1) :: Int -> Int
let g = (*2) :: Int -> Int

-- Maybe
putStrLn "--- Maybe ---"
putStrLn $ "fmap id (Just 42) = id: " ++ show (fmap id (Just 42) == (Just 42 :: Maybe Int))
putStrLn $ "fmap (f.g) = fmap f . fmap g (Just 5): " ++ show (fmap (f.g) (Just 5) == (fmap f . fmap g) (Just 5))
putStrLn $ "fmap id Nothing = id: " ++ show (fmap id (Nothing :: Maybe Int) == Nothing)

-- List
putStrLn "--- List ---"
let xs = [1,2,3::Int]
putStrLn $ "fmap id [1,2,3] = id: " ++ show (fmap id xs == xs)
putStrLn $ "fmap (f.g) = fmap f . fmap g [1,2,3]: " ++ show (fmap (f.g) xs == (fmap f . fmap g) xs)

-- Either
putStrLn "--- Either ---"
let r = Right 42 :: Either String Int
let l = Left "err" :: Either String Int
putStrLn $ "fmap id (Right 42) = id: " ++ show (fmap id r == r)
putStrLn $ "fmap (f.g) (Right 5): " ++ show (fmap (f.g) (Right 5 :: Either String Int) == (fmap f . fmap g) (Right 5 :: Either String Int))
putStrLn $ "fmap id (Left err) = id: " ++ show (fmap id l == l)

-- Function functor: (->) r
putStrLn "--- Function functor ---"
let h = fmap (+1) (*2) :: Int -> Int
putStrLn $ "fmap (+1) (*2) $ 5 = " ++ show (h 5)

## 2️⃣ `Contravariant` — функторы, обращающие стрелки

**Контравариантный функтор** отображает морфизмы в обратном направлении:

```haskell
class Contravariant f where
  contramap :: (a -> b) -> f b -> f a
```

Примеры:
- `Predicate a = a -> Bool` (предикат)
- `Comparison a = a -> a -> Ordering` (компаратор)
- `Op b a = a -> b` (функция результата)

In [ ]:
-- Contravariant class
class Contravariant f where
  contramap :: (b -> a) -> f a -> f b

newtype Predicate a = Predicate { runPredicate :: a -> Bool }
instance Contravariant Predicate where
  contramap f (Predicate p) = Predicate (p . f)

newtype Op r a = Op { getOp :: a -> r }
instance Contravariant (Op r) where
  contramap f (Op g) = Op (g . f)

newtype Comparison a = Comparison { runComparison :: a -> a -> Ordering }
instance Contravariant Comparison where
  contramap f (Comparison cmp) = Comparison (\x y -> cmp (f x) (f y))

isEven :: Predicate Int
isEven = Predicate even

isEvenLength :: Predicate String
isEvenLength = contramap length isEven

putStrLn "--- Predicate ---"
putStrLn $ "isEven 4 = " ++ show (runPredicate isEven 4)
putStrLn $ "isEven 5 = " ++ show (runPredicate isEven 5)
let s2 = "hi"
let s3 = "hey"
putStrLn $ "isEvenLength hi = " ++ show (runPredicate isEvenLength s2)
putStrLn $ "isEvenLength hey = " ++ show (runPredicate isEvenLength s3)

compareLength :: Comparison String
compareLength = contramap length (Comparison compare)
putStrLn "--- Comparison ---"
putStrLn $ "compareLength hi hey = " ++ show (runComparison compareLength s2 s3)
putStrLn $ "compareLength ab cd = " ++ show (runComparison compareLength "ab" "cd")

-- Law: contramap (f.g) = contramap g . contramap f
let f = length :: String -> Int
let g = show :: Int -> String
let left  = runPredicate (contramap (f . g) isEven) (42::Int)
let right = runPredicate ((contramap g . contramap f) isEven) (42::Int)
putStrLn $ "contramap law: " ++ show (left == right)

## 3️⃣ `Bifunctor` и `Bitraversable`

**Бифунктор** имеет два ковариантных типовых параметра:

```haskell
class Bifunctor f where
  bimap  :: (a -> c) -> (b -> d) -> f a b -> f c d
  first  :: (a -> c) -> f a b -> f c b
  second :: (b -> d) -> f a b -> f a d
```

Примеры: `(,)`, `Either`, `These`

In [ ]:
-- Bifunctor class
class Bifunctor p where
  bimap  :: (a -> c) -> (b -> d) -> p a b -> p c d
  first  :: (a -> c) -> p a b -> p c b
  first  f = bimap f id
  second :: (b -> d) -> p a b -> p a d
  second g = bimap id g

-- (,) instance
instance Bifunctor (,) where
  bimap f g (a, b) = (f a, g b)

-- Either instance
instance Bifunctor Either where
  bimap f _ (Left a)  = Left  (f a)
  bimap _ g (Right b) = Right (g b)

-- Test
putStrLn "--- Bifunctor (,) ---"
let p1 = bimap (+1) (++ "!") (3 :: Int, "hello")
putStrLn $ "bimap (+1) (++!) (3,hello) = " ++ show p1
let p2 = first (*2) (5 :: Int, True)
putStrLn $ "first (*2) (5,True) = " ++ show p2
let p3 = second not (5 :: Int, True)
putStrLn $ "second not (5,True) = " ++ show p3

putStrLn "--- Bifunctor Either ---"
let e1 = bimap length (++ "!") (Left "hello" :: Either String String)
putStrLn $ "bimap length (++!) (Left hello) = " ++ show e1
let e2 = bimap length (++ "!") (Right "world" :: Either String String)
putStrLn $ "bimap length (++!) (Right world) = " ++ show e2

-- Bitraversable: effectful bimap
bitraverse :: Applicative f => (a -> f c) -> (b -> f d) -> (a,b) -> f (c,d)
bitraverse f g (a, b) = (,) <$> f a <*> g b

-- Example: validate a pair
validatePositive :: Int -> Maybe Int
validatePositive x = if x > 0 then Just x else Nothing

validateNonEmpty :: String -> Maybe String
validateNonEmpty s = if not (null s) then Just s else Nothing

putStrLn "--- Bitraversable ---"
let r1 = bitraverse validatePositive validateNonEmpty (5::Int, "hi")
putStrLn $ "bitraverse (5,hi) = " ++ show r1
let r2 = bitraverse validatePositive validateNonEmpty (-1::Int, "hi")
putStrLn $ "bitraverse (-1,hi) = " ++ show r2
let r3 = bitraverse validatePositive validateNonEmpty (5::Int, "")
putStrLn $ "bitraverse (5,) = " ++ show r3

## 4️⃣ `Invariant` — функторы в обоих направлениях

**Инвариантный функтор** (экспоненциальный функтор) имеет оба отображения:

```haskell
class Invariant f where
  invmap :: (a -> b) -> (b -> a) -> f a -> f b
```

Обобщает как `Functor`, так и `Contravariant`. Полезен для типов, инвариантных к изоморфизмам.

In [ ]:
-- Invariant class
class Invariant f where
  invmap :: (a -> b) -> (b -> a) -> f a -> f b

data Codec a = Codec
  { encode :: a -> String
  , decode :: String -> Maybe a
  }

instance Invariant Codec where
  invmap f g (Codec enc dec) = Codec
    { encode = enc . g
    , decode = fmap f . dec
    }

-- readInt: simple parser
readInt :: String -> Maybe Int
readInt s = case reads s of
  [(x, "")] -> Just (x :: Int)
  _         -> Nothing

intCodec :: Codec Int
intCodec = Codec show readInt

data NonNeg = NonNeg Int deriving Show

nonNegCodec :: Codec NonNeg
nonNegCodec = invmap NonNeg (\(NonNeg n) -> n) intCodec

putStrLn "--- Invariant (Codec) ---"
putStrLn $ "encode intCodec 42 = " ++ encode intCodec 42
putStrLn $ "decode intCodec 42 = " ++ show (decode intCodec "42")
putStrLn $ "decode intCodec abc = " ++ show (decode intCodec "abc")
putStrLn $ "encode nonNegCodec (NonNeg 7) = " ++ encode nonNegCodec (NonNeg 7)
putStrLn $ "decode nonNegCodec 7 = " ++ show (decode nonNegCodec "7")

## 5️⃣ `Applicative` как моноидальный функтор

`Applicative` = **моноидальный функтор** `(Hask, ×, ()) -> (Hask, ×, ())`:

```haskell
class Functor f => Applicative f where
  pure  :: a -> f a
  (<*>) :: f (a -> b) -> f a -> f b
```

Законы:
- Тождество: `pure id <*> v = v`
- Композиция: `pure (.) <*> u <*> v <*> w = u <*> (v <*> w)`
- Гомоморфизм: `pure f <*> pure x = pure (f x)`
- Перестановочность: `u <*> pure y = pure ($ y) <*> u`

In [ ]:
-- Applicative laws: inline verification
let f = (+1) :: Int -> Int
let x = 42  :: Int

-- Maybe
putStrLn "--- Applicative laws (Maybe) ---"
let v = Just 42 :: Maybe Int
let law1 = (pure id <*> v) == v
putStrLn $ "identity (Just 42): " ++ show law1
let law2 = (pure f <*> (pure x :: Maybe Int)) == pure (f x)
putStrLn $ "homomorphism: " ++ show law2
let u = pure f :: Maybe (Int -> Int)
let law3 = (u <*> pure x) == (pure ($ x) <*> u)
putStrLn $ "interchange: " ++ show law3
let law4 = (pure id <*> (Nothing :: Maybe Int)) == Nothing
putStrLn $ "identity Nothing: " ++ show law4

-- List
putStrLn "--- Applicative laws (List) ---"
let vs = [1,2,3::Int]
let llaw1 = (pure id <*> vs) == vs
putStrLn $ "identity [1,2,3]: " ++ show llaw1
let llaw2 = (pure f <*> (pure x :: [Int])) == pure (f x)
putStrLn $ "homomorphism: " ++ show llaw2

-- Monoidal formulation
zipA :: Applicative g => g a -> g b -> g (a,b)
zipA fa fb = (,) <$> fa <*> fb

putStrLn "--- Monoidal Applicative ---"
let z1 = zipA (Just (1::Int)) (Just "hi")
putStrLn $ "zipA (Just 1) (Just hi) = " ++ show z1
let z2 = zipA (Nothing::Maybe Int) (Just "hi")
putStrLn $ "zipA Nothing (Just hi) = " ++ show z2
let z3 = zipA [1,2::Int] ["a","b","c"]
putStrLn $ "zipA [1,2] [a,b,c] = " ++ show z3

-- Applicative vs Monad
putStrLn "--- Applicative: static structure ---"
let sc = (,) <$> [1,2::Int] <*> [10,20::Int]
putStrLn $ "(,) <$> [1,2] <*> [10,20] = " ++ show sc

putStrLn "--- Monad: dynamic (data-dependent) ---"
let dc = [1,2::Int] >>= (\n -> map (n,) [1..n])
putStrLn $ "n <- [1,2]; k <- [1..n]; (n,k) = " ++ show dc

## 6️⃣ `Alternative` — моноид над аппликативными

`Alternative` добавляет моноидальную структуру к `Applicative`:

```haskell
class Applicative f => Alternative f where
  empty :: f a
  (<|>) :: f a -> f a -> f a
```

Законы: `empty` — нейтральный элемент, `(<|>)` — ассоциативен.

Примеры: `Maybe`, `[]`, `Parser`

In [ ]:
import Control.Applicative (Alternative(..))

putStrLn "--- Alternative Maybe ---"
putStrLn $ "Nothing <|> Just 1 = " ++ show (Nothing <|> Just (1::Int))
putStrLn $ "Just 1 <|> Just 2  = " ++ show (Just (1::Int) <|> Just 2)
putStrLn $ "Nothing <|> Nothing = " ++ show ((Nothing :: Maybe Int) <|> Nothing)

putStrLn "--- Alternative List ---"
let xs1 = [1,2::Int] <|> [3,4]
putStrLn $ "[1,2] <|> [3,4] = " ++ show xs1
let xs2 = ([] :: [Int]) <|> [1,2]
putStrLn $ "[] <|> [1,2] = " ++ show xs2

-- guard and list comprehensions
putStrLn "--- guard ---"
let evens = do { x <- [1..10::Int]; if even x then [x] else [] }
putStrLn $ "evens via guard: " ++ show evens

-- firstJust / asum
firstJust :: [Maybe a] -> Maybe a
firstJust = foldr (<|>) Nothing

let options = [Nothing, Nothing, Just (42::Int), Just 99]
putStrLn $ "firstJust options = " ++ show (firstJust options)

## 7️⃣ `Selective` — между Applicative и Monad

`Selective` (Андрей Мохов, 2019) строго между `Applicative` и `Monad`:

```haskell
class Applicative f => Selective f where
  select :: f (Either a b) -> f (a -> b) -> f b
```

Позволяет **условное выполнение**: правый операнд запускается только при необходимости.
Слабее монады (нет полного связывания), сильнее аппликатива (есть условие).

In [ ]:
import Control.Applicative (Alternative(..))

-- Selective: between Applicative and Monad
-- select: Left means apply function, Right means skip (return the b)
class Applicative f => Selective f where
  select :: f (Either a b) -> f (a -> b) -> f b

-- Monad gives Selective for free
selectM :: Monad m => m (Either a b) -> m (a -> b) -> m b
selectM mx mf = do
  ex <- mx
  case ex of
    Left  a -> fmap ($ a) mf
    Right b -> return b

instance Selective Maybe where
  select = selectM

instance Selective [] where
  select = selectM

-- Demonstration 1: select with Maybe
putStrLn "--- Selective Maybe ---"
let r1 = select (Just (Left (42::Int))) (Just (+1))
putStrLn $ "select (Just (Left 42)) (Just (+1)) = " ++ show (r1 :: Maybe Int)

let r2 = select (Just (Right (99::Int))) (Just (+1))
putStrLn $ "select (Just (Right 99)) (Just (+1)) = " ++ show (r2 :: Maybe Int)

let r3 = select (Nothing :: Maybe (Either Int Int)) (Just (+1))
putStrLn $ "select Nothing (Just (+1)) = " ++ show (r3 :: Maybe Int)

putStrLn ""
putStrLn "--- Selective []: branches ---"
let r4 = select [Left (1::Int), Right 10, Left 2] [(+100)]
putStrLn $ "select [Left 1, Right 10, Left 2] [(+100)] = " ++ show (r4 :: [Int])

putStrLn ""
putStrLn "--- Conditional effect: Left=apply, Right=skip ---"
-- Left () => apply const "yes", Right "no" => skip
let r5 = select (Just (Left ())) (Just (const "yes"))
let r6 = select (Just (Right "no")) (Just (const "yes"))
putStrLn $ "Left ()          -> " ++ show (r5 :: Maybe String)
putStrLn $ "Right 'no'       -> " ++ show (r6 :: Maybe String)

## 8️⃣ Полная иерархия

```
Functor
  ├── Contravariant
  ├── Bifunctor
  ├── Invariant
  └── Applicative
        ├── Alternative
        ├── Selective
        └── Monad
              └── MonadPlus
```

Каждый уровень добавляет структуру:
- `Functor`: только отображение
- `Applicative`: последовательные вычисления
- `Monad`: зависимые последовательные вычисления

## Диаграмма: иерархия функторов

Полная иерархия классов типов — от Functor до Monad, Alternative и сопутствующих классов.

![Иерархия функторов](../diagrams/functors/fh_hierarchy.svg)

## Итоги

| Класс | Ключевой метод | Категориальный смысл |
|-------|----------------|---------------------|
| Functor | `fmap` | Эндофунктор в Hask |
| Contravariant | `contramap` | Эндофунктор в Hask^op |
| Bifunctor | `bimap` | Функтор `Hask x Hask -> Hask` |
| Applicative | `<*>` | Моноидальный функтор |
| Alternative | `<|>` | Аппликатив с моноидом |
| Selective | `select` | Условные эффекты |
| Monad | `>>=` | Монада (алгебра над тройкой) |

---

**← Предыдущий:** [Типы как алгебра](TypeAlgebra.ipynb)  |  **Следующий →** [Foldable & Traversable](FoldableTraversable.ipynb)
